# 3주차 실습. 표를 코드로 읽기

이 파일을 직접 고치지 말고, 먼저 본인 폴더로 복사한다. Qoka 대화 입력창에 입력한다.

> course/labs/week3/week3_reading.ipynb 를 analysis/week3/ 폴더로 복사해서 열어줘.

열리면 오른쪽 위 커널을 **Qoka Run Environment**로 고른다.

## 이 노트북을 쓰는 방법

코드는 미리 적혀 있다. 셀마다 다음 순서를 따른다.

1. **말로 하면** 칸을 읽는다
2. 코드를 읽고, 실행하기 전에 **예측** 칸에 결과를 적는다
3. 셀을 실행하고 예측과 비교한다
4. `# 고칠 곳`이 있으면 그 값 하나만 바꾸고 다시 실행한다

셀은 위에서부터 순서대로 실행한다. 건너뛰면 위에서 만든 변수가 없어서 오류가 난다.

## 준비

**말로 하면**: 필요한 패키지를 설치하고 불러온 뒤, 데이터 파일 위치를 찾는다. 그대로 실행한다.

In [ ]:
%pip install -q pandas numpy matplotlib

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

root = Path.cwd()
while not (root / "data").exists() and root != root.parent:
    root = root.parent
path = root / "data" / "airway_scaledcounts.subset.tsv"
print(path.exists())

## 셀 1. 파일을 표로 읽기

**말로 하면**: 탭으로 나뉘고 소수점이 쉼표인 파일을 표로 읽는다. `ensgene` 열은 행 이름으로 쓴다. 표의 크기를 출력하고 앞 다섯 줄을 본다.

**예측**: 표의 행과 열은 각각 몇 개일까?

> (실행하기 전에 여기에 적는다)

In [ ]:
# 셀 1.
counts = pd.read_csv(path, sep="\t", decimal=",", index_col="ensgene")
print(counts.shape)
counts.head()

## 셀 2. 한 열을 요약하기

**말로 하면**: `1_control` 열의 개수, 평균, 중앙값, 최댓값을 본다.

**예측**: 평균과 중앙값(50%) 중 어느 쪽이 클까?

> (실행하기 전에 여기에 적는다)

In [ ]:
# 셀 2.
counts["1_control"].describe()

## 셀 3. 한 열의 분포 그리기

**말로 하면**: `1_control` 값을 100개 구간으로 나눠 막대 그래프를 그린다.

**예측**: 막대는 어디에 몰려 있을까?

> (실행하기 전에 여기에 적는다)

In [ ]:
# 셀 3.
plt.hist(counts["1_control"], bins=100)
plt.xlabel("1_control")
plt.ylabel("number of genes")
plt.show()

## 셀 4. 새 열 만들기

**말로 하면**: 대조군 두 샘플의 평균과 처리군 두 샘플의 평균을 새 열로 만든다.

**예측**: 열은 몇 개가 될까?

> (실행하기 전에 여기에 적는다)

In [ ]:
# 셀 4.
counts["control_mean"] = (counts["1_control"] + counts["2_control"]) / 2
counts["treated_mean"] = (counts["1_treated"] + counts["2_treated"]) / 2
print(counts.shape)
counts.head()

## 셀 5. 두 조건을 그림으로 비교하기

**말로 하면**: 대조군 평균을 가로축, 처리군 평균을 세로축으로 점을 찍는다. 두 값이 같은 자리에 점선을 긋고, 축은 0부터 2500까지만 본다.

**예측**: 점들은 대부분 점선 위에 있을까, 아래에 있을까?

> (실행하기 전에 여기에 적는다)

In [ ]:
# 셀 5.
plt.scatter(counts["control_mean"], counts["treated_mean"], s=4, alpha=0.3)
plt.plot([0, 2500], [0, 2500], color="orange", linestyle="--")
plt.xlim(0, 2500)
plt.ylim(0, 2500)
plt.xlabel("control_mean")
plt.ylabel("treated_mean")
plt.show()

## 셀 6. 배수를 log2로 계산하기

**말로 하면**: 처리군 평균을 대조군 평균으로 나누고 log2를 씌운다. 2배는 1, 8배는 3, 절반은 -1이다. 그다음 계산이 제대로 된 유전자와 안 된 유전자를 센다.

**예측**: 실행하면 오류가 날까? 계산이 제대로 되는 유전자는 38,694개 중 몇 개쯤일까?

> (실행하기 전에 여기에 적는다)

In [ ]:
# 셀 6.
counts["logexpr"] = np.log2(counts["treated_mean"] / counts["control_mean"])

print("둘 다 0 (NaN)        :", counts["logexpr"].isna().sum())
print("대조군만 0 (inf)     :", (counts["logexpr"] == np.inf).sum())
print("처리군만 0 (-inf)    :", (counts["logexpr"] == -np.inf).sum())
print("계산된 유전자       :", np.isfinite(counts["logexpr"]).sum())

## 셀 7. 8배 이상 변한 유전자 칠하기

**말로 하면**: log2 값이 3 이상이거나 -3 이하인 유전자는 빨간색, 나머지는 회색으로 칠한다. 빨간색 유전자 수와, 그중 값이 있어서 그림에 찍힐 수 있는 유전자 수를 출력하고 그림을 그린다.

**예측**: 두 수는 같을까?

> (실행하기 전에 여기에 적는다)

In [ ]:
# 셀 7.
is_red = (counts["logexpr"] >= 3) | (counts["logexpr"] <= -3)
counts["color"] = np.where(is_red, "darkred", "darkgray")

print("빨간색 유전자        :", is_red.sum())
print("그중 그림에 찍히는 것:", (is_red & np.isfinite(counts["logexpr"])).sum())

plt.figure(figsize=(8, 4))
plt.scatter(range(len(counts)), counts["logexpr"], c=counts["color"], s=4)
plt.xlabel("gene index")
plt.ylabel("log2(treated / control)")
plt.show()

## 셀 8. 배수가 큰 순서로 세우기

**말로 하면**: 계산이 된 유전자만 남기고, log2 값이 큰 순서로 줄을 세우고, 순위를 붙이고, 위 10줄을 본다.

**예측**: 맨 위 유전자들의 대조군 평균은 클까, 작을까?

> (실행하기 전에 여기에 적는다)

In [ ]:
# 셀 8.
ranked = counts[np.isfinite(counts["logexpr"])].sort_values("logexpr", ascending=False)
ranked["rank"] = range(1, len(ranked) + 1)
ranked[["control_mean", "treated_mean", "logexpr", "rank"]].head(10)

## 셀 9. 논문의 유전자 순위 찾기

2주차에 찾은 Himes et al. (2014)은 덱사메타손에 반응하는 유전자로 CRISPLD2를 보고했다. Ensembl ID는 `ENSG00000103196`이다.

**말로 하면**: 순위를 매긴 표에서 CRISPLD2의 행을 꺼낸다.

**예측**: 19,971개 중 몇 번째일까?

> (실행하기 전에 여기에 적는다)

In [ ]:
# 셀 9.
ranked.loc["ENSG00000103196", ["control_mean", "treated_mean", "logexpr", "rank"]]

## 셀 10. 발현량이 적은 유전자 걸러내기

**말로 하면**: 대조군 평균과 처리군 평균이 모두 `min_mean` 이상인 유전자만 남기고 순위를 다시 매긴다. 남은 유전자 수를 출력하고 위 10줄을 본다.

**예측**: 유전자는 몇 개가 남을까? 위 10줄의 대조군 평균은 셀 8과 어떻게 다를까?

> (실행하기 전에 여기에 적는다)

In [ ]:
# 셀 10.
min_mean = 10                                   # 고칠 곳

kept = ranked[(ranked["control_mean"] >= min_mean) & (ranked["treated_mean"] >= min_mean)].copy()
kept["rank"] = range(1, len(kept) + 1)
print(len(ranked), "->", len(kept))
kept[["control_mean", "treated_mean", "logexpr", "rank"]].head(10)

## 셀 11. 걸러낸 표에서 순위 다시 찾기

**말로 하면**: 걸러낸 표에서 CRISPLD2의 행을 꺼낸다.

**예측**: 몇 번째로 바뀔까?

> (실행하기 전에 여기에 적는다)

**고칠 곳**: 셀 10의 `min_mean`을 `50`으로 바꾸고 셀 10과 셀 11을 다시 실행한다. 유전자 수와 CRISPLD2의 순위는 어떻게 될까?

> (실행하기 전에 여기에 적는다)

In [ ]:
# 셀 11.
kept.loc["ENSG00000103196", ["control_mean", "treated_mean", "logexpr", "rank"]]